In [ ]:
!pip install mxnet

In [ ]:
!pip install huggingface_hub

In [2]:
!pip install python-dotenv

In [2]:
import numpy as np
# Manually add the missing alias back to numpy
np.bool = bool
np.float = float
np.int = int

import mxnet as mx
print("MXNet imported successfully!")

MXNet imported successfully!


In [3]:
from scipy.sparse import csr_matrix
import pandas as pd
import boto3
import json
import sagemaker
from sklearn.model_selection import train_test_split
import datetime, time
import sagemaker.amazon.common as smac
import io, os
from io import BytesIO
import requests
import tarfile, zipfile
from sklearn.metrics.pairwise import cosine_similarity
from decimal import Decimal
from boto3.dynamodb.conditions import Key
from dotenv import load_dotenv

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [4]:
bucket = "huzaifas-recommendation-engine"
# Initialize session and role
session = sagemaker.Session()
role = sagemaker.get_execution_role()

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


Load data from Glue

In [5]:
df = pd.read_parquet(f"s3://{bucket}/ready-for-recordio/")


s3_client = boto3.client('s3')
obj = s3_client.get_object(Bucket=bucket, Key='metadata/config.json')
metadata = json.loads(obj['Body'].read())

num_users = metadata['num_users']
num_items = metadata['num_items']
feature_dim = num_users + num_items

Build Sparse Feature Matrix

In [52]:

y = df['label'].values.astype("float32")
n_rows = len(df)


rows = []
cols = []
data = []

for i, feature_dict in enumerate(df['features']):
    idx = feature_dict['indices']
    rows.extend([i] * len(idx))
    cols.extend(idx)
    data.extend([1.0] * len(idx))


X = csr_matrix((data, (rows, cols)), shape=(n_rows, feature_dim))

print(f"Matrix built! Shape: {X.shape}")

Matrix built! Shape: (266792, 8003)


Splitting and Uploading

In [53]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = X_train.astype('float32')
y_train = y_train.astype('float32')
X_test = X_test.astype('float32')
y_test = y_test.astype('float32')

# Convert to RecordIO format locally
with open("train.recordio", "wb") as f:
    smac.write_spmatrix_to_sparse_tensor(f, X_train, y_train)
with open("test.recordio", "wb") as f:
    smac.write_spmatrix_to_sparse_tensor(f, X_test, y_test)

# Upload to S3
train_path = session.upload_data("train.recordio", bucket=bucket, key_prefix="train")
test_path = session.upload_data("test.recordio", bucket=bucket, key_prefix="test")

Train the Factorization Machines Model

Configure the Estimator

In [54]:
# Estimator Block
fm = sagemaker.estimator.Estimator(
    image_uri=sagemaker.image_uris.retrieve(
        "factorization-machines", session.boto_region_name
    ),
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path="s3://huzaifas-recommendation-engine/fm-output/",
    sagemaker_session=session,
    environment={}
)

# Hyperparameters Block
fm.set_hyperparameters(
    feature_dim=num_users + num_items,
    predictor_type="regressor",
    num_factors=32,
    epochs=50,
    mini_batch_size=200
)

Start Training

In [ ]:
fm.fit({
    "train": "s3://huzaifas-recommendation-engine/train/train.recordio",
    "test": "s3://huzaifas-recommendation-engine/test/test.recordio"
})

Deploy the Model to a Real-Time Sagemaker Endpoint

In [43]:
predictor = fm.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large"
)

-----------!

Generate Recommendations (Inference)

In [56]:
s3 = boto3.client("s3", region_name='us-east-1')

In [ ]:
# Config
endpoint_name = "factorization-machines-2026-04-17-14-17-23-398"
region = "us-east-1"
bucket = "huzaifas-recommendation-engine"
prefix = "inference/output"

# Clients
rt = boto3.client("sagemaker-runtime", region_name=region)

# Invoke Endpoint
with open("train.recordio", "rb") as f:
    payload = f.read()

resp = rt.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/x-recordio-protobuf",
    Accept="application/json",
    Body=payload
)

raw_response = resp["Body"].read().decode("utf-8")

# Prepare output (safe JSON)
try:
    response_json = json.loads(raw_response)
except json.JSONDecodeError:
    response_json = {"raw_response": raw_response}

output_object = {
    "endpoint": endpoint_name,
    "timestamp_utc": datetime.datetime.utcnow().isoformat(),
    "response": response_json
}

# Write to S3
key = f"{prefix}/prediction-{datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}.json"
s3.put_object(
    Bucket=bucket,
    Key=key,
    Body=json.dumps(output_object, indent=2).encode("utf-8"),
    ContentType="application/json"
)

print("Prediction saved to:")
print(f"s3://{bucket}/{key}")

Invoke

In [ ]:
# 1. Select a batch (e.g., first 500 rows of your test set)
batch_size = 500
X_batch = X_test[:batch_size].toarray().astype('float32')
y_batch = y_test[:batch_size].astype('float32')

# 2. Convert the WHOLE BATCH to one RecordIO payload
buf = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf, X_batch, y_batch)
buf.seek(0)
batch_payload = buf.read()

# 3. Invoke once
resp = rt.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/x-recordio-protobuf",
    Accept="application/json",
    Body=batch_payload
)

raw_response = resp["Body"].read().decode("utf-8")

Re-initialize Session

In [ ]:
# 1. Re-initialize session
session = sagemaker.Session()
role = sagemaker.get_execution_role()

# 2. Attach to the existing training job
job_name = "factorization-machines-2026-04-17-13-45-31-969"
fm = sagemaker.estimator.Estimator.attach(job_name)

# 3. If your endpoint is still running, attach to it too:
endpoint_name = "factorization-machines-2026-04-17-14-17-23-398"
predictor = sagemaker.predictor.Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=session
)

Extraction

In [57]:
# Step 1: Get the latest model path dynamically
model_s3_path = fm.latest_training_job.describe()['ModelArtifacts']['S3ModelArtifacts']
print(f"Latest model path: {model_s3_path}")

# Step 2: Parse bucket and key from the S3 path
# model_s3_path looks like: s3://bucket-name/some/path/model.tar.gz
path_parts = model_s3_path.replace("s3://", "").split("/", 1)
bucket_name = path_parts[0]
key = path_parts[1]
print(f"Bucket: {bucket_name}")
print(f"Key: {key}")

# Step 3: Download the latest model
s3.download_file(bucket_name, key, 'model.tar.gz')
print("Downloaded successfully!")

# Step 4: Extract tar
os.makedirs("./model_extracted", exist_ok=True)
with tarfile.open('model.tar.gz', 'r:gz') as tar:
    tar.extractall("./model_extracted")
print("Tar extracted!")

# Step 5: Extract inner zip
with zipfile.ZipFile("./model_extracted/model_algo-1", "r") as z:
    print("Files inside zip:", z.namelist())
    z.extractall("./model_extracted_inner")
print("Zip extracted!")

# Step 6: Verify
for f in os.listdir("./model_extracted_inner"):
    full = os.path.join("./model_extracted_inner", f)
    print(f"{f}: {os.path.getsize(full)} bytes")

Latest model path: s3://huzaifas-recommendation-engine/fm-output/factorization-machines-2026-04-20-12-24-28-562/output/model.tar.gz
Bucket: huzaifas-recommendation-engine
Key: fm-output/factorization-machines-2026-04-20-12-24-28-562/output/model.tar.gz
Downloaded successfully!
Tar extracted!
Files inside zip: ['meta.json', 'symbol.json', 'params']
Zip extracted!
meta.json: 839 bytes
params: 1184719 bytes
symbol.json: 3070 bytes


In [6]:
# Step 1: Check the raw file header (magic bytes)
with open("./model_extracted/model_algo-1", "rb") as f:
    header = f.read(16)
    print("Raw header bytes:", header)
    print("Hex:", header.hex())

print("Is zip?", zipfile.is_zipfile("./model_extracted/model_algo-1"))
print("Is tar?", tarfile.is_tarfile("./model_extracted/model_algo-1"))

Raw header bytes: b'PK\x03\x04\x14\x00\x00\x00\x00\x00\xcbc\x94\\\x8a\x84'
Hex: 504b0304140000000000cb63945c8a84
Is zip? True
Is tar? False


In [7]:
# Extract the zip
with zipfile.ZipFile("./model_extracted/model_algo-1", "r") as z:
    print("Files inside zip:")
    for name in z.namelist():
        print(f"  {name}")
    z.extractall("./model_extracted_inner")

Files inside zip:
  meta.json
  symbol.json
  params


Extract Params

In [8]:
# Step 1: Check meta.json to understand the model structure
with open("./model_extracted_inner/meta.json", "r") as f:
    meta = json.load(f)
print("Meta:", json.dumps(meta, indent=2))

# Step 2: Load the params file
params = mx.nd.load("./model_extracted_inner/params")

# Step 3: Inspect all keys and shapes
print("\nKeys found:")
for key in params.keys():
    print(f"  {key}: {params[key].shape}")

Meta: {
  "version": 1,
  "epoch_number": 1,
  "training_parameters": {
    "epochs": "50",
    "mini_batch_size": "200",
    "use_bias": "true",
    "use_linear": "true",
    "bias_lr": "0.1",
    "linear_lr": "0.001",
    "factors_lr": "0.0001",
    "bias_wd": "0.01",
    "linear_wd": "0.001",
    "factors_wd": "0.00001",
    "bias_init_method": "normal",
    "bias_init_sigma": "0.01",
    "linear_init_method": "normal",
    "linear_init_sigma": "0.01",
    "factors_init_method": "normal",
    "factors_init_sigma": "0.001",
    "batch_metrics_publish_interval": "500",
    "_data_format": "record",
    "_kvstore": "auto",
    "_learning_rate": "1.0",
    "_log_level": "info",
    "_num_gpus": "auto",
    "_num_kv_servers": "auto",
    "_optimizer": "adam",
    "_tuning_objective_metric": "",
    "_use_full_symbolic": "true",
    "_wd": "1.0",
    "feature_dim": "8003",
    "num_factors": "32",
    "predictor_type": "regressor"
  },
  "label_names": [
    "out_label"
  ]
}

Keys found:

Extract Embeddings

In [9]:
# Extract all parameter matrices
v = params["arg:v"].asnumpy()          # (8003, 10) - latent factor embeddings
w1 = params["arg:w1_weight"].asnumpy() # (8003, 1)  - linear weights
w0 = params["arg:w0_weight"].asnumpy() # (1,)        - global bias

print(f"Full embedding matrix shape: {v.shape}")

# --- Slice users and items ---
# feature_dim = 8003, which is num_users + num_items
# You need to know num_users to split correctly
num_users = 1000   # <-- replace with your actual number
num_items = 8003 - num_users

user_embeddings = v[:num_users]         # (num_users, 10)
item_embeddings = v[num_users:]         # (num_items, 10)

print(f"User embeddings: {user_embeddings.shape}")
print(f"Item embeddings: {item_embeddings.shape}")

# --- Save for later use ---
np.save("user_embeddings.npy", user_embeddings)
np.save("item_embeddings.npy", item_embeddings)
print("Saved successfully!")

Full embedding matrix shape: (8003, 32)
User embeddings: (1000, 32)
Item embeddings: (7003, 32)


Saved successfully!


In [10]:
# Load your original dataset to get exact counts
df = pd.read_csv("data.csv", encoding="latin-1")

num_users = df["CustomerID"].nunique()
num_items = df["StockCode"].nunique()

print(f"num_users: {num_users}")
print(f"num_items: {num_items}")
print(f"Total (should equal 8003): {num_users + num_items}")

num_users: 4372
num_items: 4070
Total (should equal 8003): 8442


In [11]:
# Recreate the EXACT same encoding used in your Glue job
# The order matters — must match how indices were assigned during training
unique_users = sorted(df["CustomerID"].unique())  # or unsorted, must match Glue
unique_items = sorted(df["StockCode"].unique())

# Build lookup dictionaries
user2idx = {u: i for i, u in enumerate(unique_users)}
item2idx = {item: i + len(unique_users) for i, item in enumerate(unique_items)}

print(f"Encoder num_users: {len(unique_users)}")
print(f"Encoder num_items: {len(unique_items)}")
print(f"Total: {len(unique_users) + len(unique_items)}")

Encoder num_users: 4373
Encoder num_items: 4070
Total: 8443


In [12]:
# Get interaction counts — model likely dropped low-interaction users/items
user_counts = df["CustomerID"].value_counts()
item_counts = df["StockCode"].value_counts()

# Try filtering by minimum interactions (common SageMaker default is 1+)
# Adjust min_interactions until you get exactly 8003
for min_interactions in [1, 2, 3, 4, 5]:
    valid_users = user_counts[user_counts >= min_interactions].index
    valid_items = item_counts[item_counts >= min_interactions].index
    total = len(valid_users) + len(valid_items)
    print(f"min_interactions={min_interactions}: users={len(valid_users)}, items={len(valid_items)}, total={total}")

min_interactions=1: users=4372, items=4070, total=8442
min_interactions=2: users=4293, items=3837, total=8130
min_interactions=3: users=4234, items=3717, total=7951
min_interactions=4: users=4181, items=3611, total=7792
min_interactions=5: users=4126, items=3529, total=7655


In [13]:
# Check if item count is stable across thresholds
# min_interactions=1 gave 4070 items
# Does that match across your Glue job output?

# Run this to see your actual item index range
print(f"If num_items = 4070, they occupy rows 3933 to 8002")
print(f"That means num_users = 8003 - 4070 = {8003 - 4070}")

# Extract with this assumption
num_users_assumed = 8003 - 4070  # = 3933
v = params["arg:v"].asnumpy()

item_embeddings = v[num_users_assumed:]
print(f"Item embeddings shape: {item_embeddings.shape}")  # should be (4070, 10)

If num_items = 4070, they occupy rows 3933 to 8002
That means num_users = 8003 - 4070 = 3933
Item embeddings shape: (4070, 32)


In [14]:
num_users = metadata["num_users"]
num_items = metadata["num_items"]

v = params["arg:v"].asnumpy()
user_embeddings = v[:num_users]
item_embeddings = v[num_users:]

print(f"num_users: {num_users}")
print(f"num_items: {num_items}")
print(f"Total: {num_users + num_items}")  # should be 8003

num_users: 4338
num_items: 3665
Total: 8003


In [15]:
# Slice the v matrix correctly
v = params["arg:v"].asnumpy()

user_embeddings = v[:num_users]   # (4338, 10)
item_embeddings = v[num_users:]   # (3665, 10)

print(f"User embeddings: {user_embeddings.shape}")  # (4338, 10)
print(f"Item embeddings: {item_embeddings.shape}")  # (3665, 10)

# Save
np.save("user_embeddings.npy", user_embeddings)
np.save("item_embeddings.npy", item_embeddings)
print("Saved successfully!")

User embeddings: (4338, 32)
Item embeddings: (3665, 32)


Saved successfully!


Build index → ID mappings

In [16]:
# Apply the same cleaning your Glue script did
df = df.dropna(subset=["CustomerID"])
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

# Recreate StringIndexer behavior
# PySpark's StringIndexer sorts by FREQUENCY (most frequent = index 0)
user_counts = df["CustomerID"].value_counts()  # sorted by frequency
item_counts = df["StockCode"].value_counts()   # sorted by frequency

# Build index → ID mappings
idx2user = {i: uid for i, uid in enumerate(user_counts.index)}
idx2item = {i: sid for i, sid in enumerate(item_counts.index)}

# Build ID → index mappings (for lookups)
user2idx = {uid: i for i, uid in idx2user.items()}
item2idx = {sid: i for i, sid in idx2item.items()}

print(f"Users mapped: {len(user2idx)}")
print(f"Items mapped: {len(item2idx)}")

Users mapped: 4338
Items mapped: 3665


In [17]:
# Verify: index 0 should be the MOST purchased item
most_frequent_item = item_counts.index[0]  # highest frequency
most_frequent_user = user_counts.index[0]

print(f"Most frequent StockCode: '{most_frequent_item}' → should be index 0")
print(f"Most frequent CustomerID: '{most_frequent_user}' → should be index 0")

# Cross check
print(f"\nitem2idx['{most_frequent_item}'] = {item2idx[most_frequent_item]}")  # should be 0
print(f"user2idx['{most_frequent_user}'] = {user2idx[most_frequent_user]}")  # should be 0

# Also check counts to build confidence
print(f"\nTop 5 most frequent items:")
print(item_counts.head())
print(f"\nTop 5 most frequent users:")
print(user_counts.head())

Most frequent StockCode: '85123A' → should be index 0
Most frequent CustomerID: '17841.0' → should be index 0

item2idx['85123A'] = 0
user2idx['17841.0'] = 0

Top 5 most frequent items:
StockCode
85123A    2035
22423     1723
85099B    1618
84879     1408
47566     1396
Name: count, dtype: int64

Top 5 most frequent users:
CustomerID
17841.0    7847
14911.0    5675
14096.0    5111
12748.0    4595
14606.0    2700
Name: count, dtype: int64


Dynamo DB Storage

Build Item-Item Similarity

In [18]:
# Compute cosine similarity between all items
# This gives a (3665, 3665) matrix
similarity_matrix = cosine_similarity(item_embeddings)

def get_similar_items(stockcode, top_n=5):
    if stockcode not in item2idx:
        print(f"Item '{stockcode}' not found in training data")
        return []
    
    idx = item2idx[stockcode]
    scores = similarity_matrix[idx]
    similar_indices = np.argsort(scores)[::-1][1:top_n+1]
    
    results = []
    for similar_idx in similar_indices:
        results.append({
            "stockcode": idx2item[similar_idx],
            "similarity_score": round(float(scores[similar_idx]), 8)  # 8 decimal places
        })
    
    return results

# Test with a few different items
for test_item in ["85123A", "22423"]:
    similar = get_similar_items(test_item, top_n=5)
    print(f"\nItems similar to '{test_item}':")
    for item in similar:
        print(f"  {item['stockcode']}: {item['similarity_score']}")


Items similar to '85123A':
  20713: 0.9524703
  23302: 0.94770491
  22385: 0.93711191
  85141: 0.93682384
  21258: 0.93270588

Items similar to '22423':
  22089: 0.97583658
  20846: 0.97086132
  84987: 0.96681714
  23297: 0.96484399
  22592: 0.96379995


In [19]:
dynamodb = boto3.resource('dynamodb', region_name='us-east-1')

Create DynamoDB Table

In [37]:
table = dynamodb.create_table(
    TableName='item-similarities',
    KeySchema=[
        {'AttributeName': 'stockcode', 'KeyType': 'HASH'}  # Partition key
    ],
    AttributeDefinitions=[
        {'AttributeName': 'stockcode', 'AttributeType': 'S'}
    ],
    BillingMode='PAY_PER_REQUEST'  # no need to provision capacity
)

table.wait_until_exists()
print("Table created successfully!")

Table created successfully!


Store All Item Similarities

In [23]:
table = dynamodb.Table('item-similarities')

# Batch write all items
with table.batch_writer() as batch:
    for stockcode, idx in item2idx.items():
        similar_items = get_similar_items(stockcode, top_n=10)
        
        batch.put_item(Item={
            'stockcode': str(stockcode),
            'similar_items': [
                {
                    'stockcode': s['stockcode'],
                    'score': Decimal(str(s['similarity_score']))
                }
                for s in similar_items
            ]
        })

print(f"Stored similarities for {len(item2idx)} items!")

Stored similarities for 3665 items!


User Similarities

In [ ]:
dynamodb = boto3.resource('dynamodb', region_name='us-east-1')

# Create new table for user similarities
table = dynamodb.create_table(
    TableName='user-similarities',
    KeySchema=[
        {'AttributeName': 'customerid', 'KeyType': 'HASH'}
    ],
    AttributeDefinitions=[
        {'AttributeName': 'customerid', 'AttributeType': 'S'}
    ],
    BillingMode='PAY_PER_REQUEST'
)
table.wait_until_exists()
print("Table created!")

Store All User similarities

In [22]:
# Compute user-user similarity matrix
user_similarity_matrix = cosine_similarity(user_embeddings)

def get_similar_users(user_idx, top_n=10):
    scores = user_similarity_matrix[user_idx]
    similar_indices = np.argsort(scores)[::-1][1:top_n+1]
    return [
        {
            "customerid": str(idx2user[i]),
            "score": Decimal(str(round(float(scores[i]), 8)))
        }
        for i in similar_indices
    ]

# Store in DynamoDB
table = dynamodb.Table('user-similarities')

with table.batch_writer() as batch:
    for customerid, idx in user2idx.items():
        similar_users = get_similar_users(idx, top_n=10)
        batch.put_item(Item={
            'customerid': str(customerid),
            'similar_users': similar_users
        })

print(f"Stored similarities for {len(user2idx)} users!")

Stored similarities for 4338 users!


Test a Lookup

In [24]:
# Fetch similar items for a given StockCode
response = table.get_item(Key={'stockcode': '85123A'})
item = response['Item']

print(f"Similar items to 85123A:")
for s in item['similar_items']:
    print(f"  {s['stockcode']}: {s['score']}")

Similar items to 85123A:
  20713: 0.9524703
  23302: 0.94770491
  22385: 0.93711191
  85141: 0.93682384
  21258: 0.93270588
  84836: 0.93061996
  22966: 0.93056488
  22596: 0.93023551
  22045: 0.92986655
  22568: 0.92723793


In [25]:
# Check raw similarity scores before rounding
idx = item2idx["85123A"]
scores = similarity_matrix[idx]

# Sort descending
sorted_scores = np.sort(scores)[::-1]
print("Top 10 raw similarity scores:")
print(sorted_scores[:10])

print("\nBottom 5 raw similarity scores:")
print(sorted_scores[-5:])

print(f"\nMax: {scores.max():.6f}")
print(f"Min: {scores.min():.6f}")
print(f"Mean: {scores.mean():.6f}")
print(f"Std: {scores.std():.6f}")

Top 10 raw similarity scores:
[1.0000001  0.9524703  0.9477049  0.9371119  0.93682384 0.9327059
 0.93061996 0.9305649  0.9302355  0.92986655]

Bottom 5 raw similarity scores:
[-0.910664   -0.913243   -0.9147301  -0.91924226 -0.92284775]

Max: 1.000000
Min: -0.922848
Mean: 0.071376
Std: 0.402283


In [26]:


api_url = "https://j1cepmxrb3.execute-api.us-east-1.amazonaws.com/prod/similar-items"

response = requests.get(api_url, params={"stockcode": "85123A"})
print(response.json())

{'statusCode': 200, 'headers': {'Access-Control-Allow-Origin': '*'}, 'body': '{"stockcode": "85123A", "similar_items": [{"score": 0.9524703, "stockcode": "20713"}, {"score": 0.94770491, "stockcode": "23302"}, {"score": 0.93711191, "stockcode": "22385"}, {"score": 0.93682384, "stockcode": "85141"}, {"score": 0.93270588, "stockcode": "21258"}, {"score": 0.93061996, "stockcode": "84836"}, {"score": 0.93056488, "stockcode": "22966"}, {"score": 0.93023551, "stockcode": "22596"}, {"score": 0.92986655, "stockcode": "22045"}, {"score": 0.92723793, "stockcode": "22568"}]}'}


In [36]:
res = requests.get("https://j1cepmxrb3.execute-api.us-east-1.amazonaws.com/prod/catalogue")
data = res.json()
catalogue_list = json.loads(data['body'])
print(catalogue_list[:3])

[{'StockCode': '10002', 'description': 'INFLATABLE POLITICAL GLOBE ', 'price': 0.85}, {'StockCode': '10080', 'description': 'GROOVY CACTUS INFLATABLE', 'price': 0.41}, {'StockCode': '10120', 'description': 'DOGGY RUBBER', 'price': 0.21}]


In [34]:
print(type(data))  # This will likely show <class 'dict'>
print(data.keys()) # See what keys are available

<class 'dict'>
dict_keys(['statusCode', 'headers', 'body'])


Build a product catalogue endpoint

In [27]:
# One row per StockCode — keep description and average price
catalogue = (
    df.groupby("StockCode")
    .agg(description=("Description", "first"), price=("UnitPrice", "mean"))
    .reset_index()
)
catalogue["price"] = catalogue["price"].round(2)

# Save as JSON and upload to S3
s3 = boto3.client("s3")
s3.put_object(
    Bucket="huzaifas-recommendation-engine",
    Key="metadata/catalogue.json",
    Body=catalogue.to_json(orient="records"),
    ContentType="application/json"
)
print(f"Uploaded {len(catalogue)} products")

Uploaded 3665 products


In [37]:
import requests
res = requests.get("https://j1cepmxrb3.execute-api.us-east-1.amazonaws.com/prod/catalogue")
print(res.headers)

{'Date': 'Mon, 20 Apr 2026 19:29:14 GMT', 'Content-Type': 'application/json', 'Content-Length': '349396', 'Connection': 'keep-alive', 'x-amzn-RequestId': '97a0654e-3463-4b28-936d-3a23791ad52f', 'Access-Control-Allow-Origin': '*', 'x-amz-apigw-id': 'cIi1gE28oAMED4g=', 'X-Amzn-Trace-Id': 'Root=1-69e67e89-4af756616cf59c084ed5f8c3;Parent=1b4da3419da353dc;Sampled=0;Lineage=1:7aa38f47:0'}


In [39]:
import requests
res = requests.get("https://j1cepmxrb3.execute-api.us-east-1.amazonaws.com/prod/catalogue")
print(res.status_code)
print(type(res.json()))
data = res.json()
catalogue_list = json.loads(data['body'])
print(catalogue_list[:3])

200
<class 'dict'>
[{'StockCode': '10002', 'description': 'INFLATABLE POLITICAL GLOBE ', 'price': 0.85}, {'StockCode': '10080', 'description': 'GROOVY CACTUS INFLATABLE', 'price': 0.41}, {'StockCode': '10120', 'description': 'DOGGY RUBBER', 'price': 0.21}]


In [40]:
res = requests.get("https://j1cepmxrb3.execute-api.us-east-1.amazonaws.com/prod/catalogue")
data = res.json()
print(type(data))
print(list(data.keys()))

<class 'dict'>
['statusCode', 'headers', 'body']


In [2]:
import boto3
import json
import base64
import pandas as pd
from io import BytesIO
import time

# Clients
s3 = boto3.client("s3")
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

BUCKET = "huzaifas-recommendation-engine"

# Step 1: Load catalogue
res = s3.get_object(Bucket=BUCKET, Key="metadata/catalogue.json")
catalogue = json.loads(res["Body"].read())
print(f"Loaded {len(catalogue)} products")

Loaded 3665 products


In [4]:
load_dotenv()

True

Image generation using Amazon Titan Image Generator

In [ ]:
# Step 2: Generate prompt
def create_prompt(description):
    return f"High quality ecommerce product photo of {description.lower().strip()}, clean white background, studio lighting, professional photography"


# Step 3: Generate and store image for one product
def generate_and_store_image(stock_code, description):
    prompt = create_prompt(description)

    body = json.dumps({
        "taskType": "TEXT_IMAGE",
        "textToImageParams": {
            "text": prompt
        },
        "imageGenerationConfig": {
            "numberOfImages": 1,
            "width": 512,
            "height": 512,
            "quality": "standard",
            "cfgScale": 8.0
        }
    })

    try:
        response = bedrock.invoke_model(
            modelId="amazon.titan-image-generator-v2:0",
            body=body,
            contentType="application/json",
            accept="application/json"
        )

        result = json.loads(response["body"].read())
        image_b64 = result["images"][0]
        image_data = base64.b64decode(image_b64)

        # Upload to S3
        key = f"product-images/{stock_code}.png"
        s3.put_object(
            Bucket=BUCKET,
            Key=key,
            Body=image_data,
            ContentType="image/png"
        )

        print(f"✅ {stock_code}: {description[:40]}")
        return True

    except Exception as e:
        print(f"❌ {stock_code}: {e}")
        return False


# Test with first 10 only
for product in catalogue[:10]:
    generate_and_store_image(product["StockCode"], product["description"])
    time.sleep(1)

Image generation using Unsplash

In [11]:
s3 = boto3.client("s3")
BUCKET = "huzaifas-recommendation-engine"
UNSPLASH_ACCESS_KEY = os.getenv("UNSPLASH_KEY")


def fetch_and_store_image(stock_code, description):
    try:
        # Search Unsplash for the product
        search_query = description.lower().strip()

        response = requests.get(
            "https://api.unsplash.com/search/photos",
            params={
                "query": search_query,
                "per_page": 1,
                "orientation": "squarish"
            },
            headers={"Authorization": f"Client-ID {UNSPLASH_ACCESS_KEY}"}
        )

        data = response.json()

        if not data["results"]:
            print(f"⚠️ {stock_code}: No image found for '{description[:40]}'")
            return False

        # Get the small image URL (400x400 approx)
        image_url = data["results"][0]["urls"]["small"]

        # Download the image
        image_response = requests.get(image_url)
        image_data = image_response.content

        # Upload to S3
        key = f"product-images/{stock_code}.jpg"
        s3.put_object(
            Bucket=BUCKET,
            Key=key,
            Body=image_data,
            ContentType="image/jpeg"
        )

        print(f"✅ {stock_code}: {description[:40]}")
        return True

    except Exception as e:
        print(f"❌ {stock_code}: {e}")
        return False


# Test with first 5
for product in catalogue[:3]:
    fetch_and_store_image(product["StockCode"], product["description"])

✅ 10002: INFLATABLE POLITICAL GLOBE 
✅ 10080: GROOVY CACTUS INFLATABLE
⚠️ 10120: No image found for 'DOGGY RUBBER'


Image generation using Hugging Face black-forest-labs

In [ ]:
from huggingface_hub import InferenceClient

s3 = boto3.client("s3")
BUCKET = "huzaifas-recommendation-engine"
HF_API_KEY = os.getenv("HF_API_KEY")
client = InferenceClient(api_key=HF_API_KEY)


def create_prompt(description):
    return f"High quality ecommerce product photo of {description.lower().strip()}, clean white background, studio lighting, professional photography"


def generate_and_store_image(stock_code, description):
    try:
        prompt = create_prompt(description)

        # Returns a PIL Image directly
        image = client.text_to_image(
            prompt=prompt,
            model="black-forest-labs/FLUX.1-dev"
        )

        # Convert PIL image to bytes
        from io import BytesIO
        buffer = BytesIO()
        image.save(buffer, format="PNG")
        image_data = buffer.getvalue()

        # Upload to S3
        s3.put_object(
            Bucket=BUCKET,
            Key=f"product-images/{stock_code}.png",
            Body=image_data,
            ContentType="image/png"
        )
        print(f"✅ {stock_code}: {description[:40]}")
        return True

    except Exception as e:
        print(f"❌ {stock_code}: {e}")
        return False


# Test with 3 first
for product in catalogue[:3]:
    generate_and_store_image(product["StockCode"], product["description"])
    time.sleep(5)

Image generation using Pollinaton AI

In [ ]:
s3 = boto3.client("s3")
BUCKET = "huzaifas-recommendation-engine"


def create_prompt(description):
    return (
        f"High quality ecommerce product photo of {description.lower().strip()}, "
        f"clean white background, studio lighting, professional photography, "
        f"sharp focus, highly detailed"
    )


def generate_and_store_image(stock_code, description):
    try:
        prompt = create_prompt(description)

        # Pollinations AI - completely free, no API key needed
        url = f"https://image.pollinations.ai/prompt/{requests.utils.quote(prompt)}?width=512&height=512&nologo=true"

        response = requests.get(url, timeout=60)

        if response.status_code == 200:
            s3.put_object(
                Bucket=BUCKET,
                Key=f"product-images/{stock_code}.png",
                Body=response.content,
                ContentType="image/png"
            )
            print(f"✅ {stock_code}: {description[:40]}")
            return True
        else:
            print(f"❌ {stock_code}: {response.status_code}")
            return False

    except Exception as e:
        print(f"❌ {stock_code}: {e}")
        return False


failed_products = []

for i, product in enumerate(catalogue):
    success = generate_and_store_image(product["StockCode"], product["description"])

    if not success:
        failed_products.append(product)

    # Progress update every 50 products
    if (i + 1) % 50 == 0:
        print(f"Progress: {i+1}/3665 products done...")
        time.sleep(60)  # respect rate limit
    else:
        time.sleep(8)

# Save failed ones to retry later
with open("failed_products.json", "w") as f:
    json.dump(failed_products, f)

print(f"Done! Failed: {len(failed_products)} products")

Cloudfare AI Image generation

In [ ]:
s3 = boto3.client("s3")
BUCKET = "huzaifas-recommendation-engine"

CF_ACCOUNT_ID = os.getenv("CF_ACCOUNT_ID")
CF_API_TOKEN = os.getenv("CF_API_TOKEN")


def create_prompt(description):
    return (
        f"High quality ecommerce product photo of {description.lower().strip()}, "
        f"clean white background, studio lighting, professional photography, "
        f"sharp focus, highly detailed"
    )

def generate_and_store_image(stock_code, description, retries=3):
    for attempt in range(retries):
        try:
            prompt = create_prompt(description)

            response = requests.post(
                f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/ai/run/@cf/stabilityai/stable-diffusion-xl-base-1.0",
                headers={
                    "Authorization": f"Bearer {CF_API_TOKEN}",
                    "Content-Type": "application/json"
                },
                json={
                    "prompt": prompt,
                    "negative_prompt": "blurry, low quality, watermark, text, people",
                    "num_steps": 20,
                    "guidance": 7.5,
                    "width": 512,
                    "height": 512
                },
                timeout=120
            )

            if response.status_code == 200:
                # Cloudflare returns raw image bytes directly
                s3.put_object(
                    Bucket=BUCKET,
                    Key=f"product-images/{stock_code}.png",
                    Body=response.content,
                    ContentType="image/png"
                )
                print(f"✅ {stock_code}: {description[:40]}")
                return True

            elif response.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"⏳ Rate limited, waiting {wait}s...")
                time.sleep(wait)

            else:
                print(f"❌ {stock_code}: {response.status_code} - {response.text[:100]}")
                return False

        except Exception as e:
            print(f"❌ {stock_code}: {e}")
            return False

    return False


failed_products = []

for i, product in enumerate(catalogue):
    success = generate_and_store_image(product["StockCode"], product["description"])

    if not success:
        failed_products.append(product)

    # Progress update every 50 products
    if (i + 1) % 50 == 0:
        print(f"Progress: {i+1}/3665 done. Failed so far: {len(failed_products)}")

    time.sleep(2)  # Cloudflare is more reliable so 2s is fine

# Save failed ones
with open("failed_products.json", "w") as f:
    json.dump(failed_products, f)

print(f"\nDone! Total failed: {len(failed_products)}")

Test

In [6]:
# Quick test to verify credentials
CF_ACCOUNT_ID = os.getenv("CF_ACCOUNT_ID")
CF_API_TOKEN = os.getenv("CF_API_TOKEN")
response = requests.get(
    f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/ai/models/search",
    headers={"Authorization": f"Bearer {CF_API_TOKEN}"}
)
print(response.status_code)

200


In [28]:
import boto3
import pandas as pd
import json
from io import StringIO

s3 = boto3.client("s3")
BUCKET = "huzaifas-recommendation-engine"

# Step 1: Read all part files and combine
dfs = []
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="processed/")

for obj in response["Contents"]:
    key = obj["Key"]
    if key.endswith(".csv"):
        res = s3.get_object(Bucket=BUCKET, Key=key)
        content = res["Body"].read().decode("utf-8")
        df = pd.read_csv(StringIO(content), header=None, names=["interaction", "user", "item"])
        dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(df)}")
print(df.head())

Total rows: 266792
   interaction    user    item
0          5.0   215.0  1427.0
1          5.0   351.0  1494.0
2          5.0   351.0  1890.0
3          5.0  2204.0   887.0
4          5.0    96.0   192.0


In [29]:
# Step 2: Map indices back to actual IDs
df["customer_id"] = df["user"].apply(lambda x: str(idx2user[int(x)]))
df["stock_code"] = df["item"].apply(lambda x: str(idx2item[int(x)]))

print(df[["customer_id", "stock_code", "interaction"]].head())
print(f"Unique users: {df['customer_id'].nunique()}")
print(f"Unique items: {df['stock_code'].nunique()}")

  customer_id stock_code  interaction
0     13269.0      23269          5.0
1     17191.0      84969          5.0
2     17191.0     85036B          5.0
3     17499.0      23371          5.0
4     17365.0      22750          5.0
Unique users: 4338
Unique items: 3665


In [30]:
# Step 3: Build user -> list of purchased items lookup
user_items = (
    df.groupby("customer_id")["stock_code"]
    .apply(list)
    .to_dict()
)

# Step 4: Upload to S3
s3.put_object(
    Bucket=BUCKET,
    Key="metadata/user_items.json",
    Body=json.dumps(user_items),
    ContentType="application/json"
)
print(f"Uploaded purchase history for {len(user_items)} users!")

Uploaded purchase history for 4338 users!


In [ ]:
import requests

res = requests.get(
    "https://j1cepmxrb3.execute-api.us-east-1.amazonaws.com/prod/user-recommendations",
    params={"user_id": "17841.0"}
)
print(res.json())